# clipsmith — Local MP4

Process a video file you already have on disk (or upload to Colab).

**What this notebook does:**
1. Installs clipsmith (CPU-only — no GPU required)
2. Transcribes the video with faster-whisper
3. Selects the best moments with an LLM (Anthropic or OpenAI)
4. Cuts final clips with ffmpeg
5. Lets you preview them inline

**Requirements:**
- Add `ANTHROPIC_API_KEY` **or** `OPENAI_API_KEY` to Colab Secrets (🔑 left panel)
- Upload your MP4 in Cell 3, or set `MP4_PATH` to an existing path

In [ ]:
# @title 1. Install clipsmith
!pip install -q clipsmith-ai
!ffmpeg -version | head -1

In [ ]:
# @title 2. Load API keys from Colab Secrets
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY") or ""
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or ""

if not os.environ["ANTHROPIC_API_KEY"] and not os.environ["OPENAI_API_KEY"]:
    raise RuntimeError("Add ANTHROPIC_API_KEY or OPENAI_API_KEY to Colab Secrets")

provider = "anthropic" if os.environ["ANTHROPIC_API_KEY"] else "openai"
print(f"Using provider: {provider}")

In [ ]:
# @title 3. Upload your MP4 (or set MP4_PATH to an existing path)
from google.colab import files
from pathlib import Path

# Option A: upload a file now
uploaded = files.upload()
MP4_PATH = next(iter(uploaded))

# Option B: point to an existing path (comment out Option A above)
# MP4_PATH = "/content/drive/MyDrive/my_stream.mp4"

print(f"MP4: {MP4_PATH}  ({Path(MP4_PATH).stat().st_size / 1e6:.0f} MB)")

In [ ]:
# @title 4. Configure
from pathlib import Path

LANGUAGE = "es"  # ISO 639-1 — change to 'en' for English streams
WHISPER_MODEL = "small"  # tiny | small | medium | large-v3

config_yaml = f"""\
work_dir: /content/work
out_dir:  /content/out

llm:
  provider: {provider}
  model_anthropic: claude-haiku-4-5-20251001
  model_openai: gpt-4.1-mini

transcribe:
  model: {WHISPER_MODEL}
  compute_type: int8
  language: {LANGUAGE}

clip:
  min_seconds: 15
  max_seconds: 30
  preroll_s: 25
  postroll_s: 10

reframe:
  mode: none

caption:
  enabled: false
"""

Path("/content/work").mkdir(parents=True, exist_ok=True)
Path("/content/out").mkdir(parents=True, exist_ok=True)
Path("config.yaml").write_text(config_yaml)
print("Config written.")

In [ ]:
# @title 5. Run the full pipeline
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "clipsmith", "process", MP4_PATH, "--config", "config.yaml", "-v"],
    check=True,
)

In [ ]:
# @title 6. Preview clips inline
from IPython.display import Video, display
from pathlib import Path

clip_dir = Path("/content/out") / Path(MP4_PATH).stem
clips = sorted(clip_dir.glob("clip_*.mp4"))

if not clips:
    print(f"No clips found in {clip_dir}. Did step 5 complete successfully?")
else:
    print(f"Found {len(clips)} clip(s):")
    for mp4 in clips:
        print(f"  {mp4.name}")
        display(Video(str(mp4), embed=True, width=360))

In [ ]:
# @title 7. Download clips to your computer
from google.colab import files
from pathlib import Path

for mp4 in sorted(Path("/content/out").rglob("clip_*.mp4")):
    files.download(str(mp4))